In [39]:
%pip install openai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [40]:
import os
import pandas as pd
import time
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# .env laden
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env", override=True)
        break

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip().strip('"').strip("'")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY fehlt! Bitte in .env Datei eintragen.")

client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI Client bereit.")

OpenAI Client bereit.


In [41]:
# Evidence-Daten laden
evidence_df = pd.read_csv("data/evidence_topic_generation.csv")
print("Evidence-Zeilen:", len(evidence_df))
print(evidence_df["doc_variant"].value_counts())

Evidence-Zeilen: 1128
doc_variant
rel            376
non_rel        376
contrastive    376
Name: count, dtype: int64


In [42]:
RUN_CONFIG = {
    "model_provider": "openai",
    "model": "gpt-4o-mini",
    "query_mode": "first",
    "doc_variant": "contrastive",            # Aendern fuer jeden Durchlauf: "rel", "non_rel", "contrastive"
    "tfidf_top_n": 5,
    "run_name": "openai_first_contrastive_tfidf5"   # Aendern fuer jeden Durchlauf
}

In [43]:
def generate_topic(
    evidence: str,
    model_provider: str = "openai",
    model: str = "gpt-4o-mini",
    retries: int = 3
) -> dict:

    system_prompt = (
        "You are an expert information retrieval analyst. "
        "Create a TREC-style search topic based on the provided evidence. "
        "Return ONLY valid JSON with exactly these keys: title, description, narrative. "
        "title: max 8 words. "
        "description: 1-2 sentences explaining what the searcher wants. "
        "narrative: 2-3 sentences defining which documents are relevant or not relevant."
    )

    user_prompt = f"Evidence:{evidence}"

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
                temperature=0.3,
                max_tokens=350,
            )
            raw = resp.choices[0].message.content.strip()

            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                title = str(data.get("title", "")).strip()
                desc  = str(data.get("description", "")).strip()
                narr  = data.get("narrative", "")
                if isinstance(narr, list):
                    narr = " ".join(str(x) for x in narr)
                narr = str(narr).strip()
                if desc and narr:
                    return {"title": title, "description": desc,
                            "narrative": narr, "raw_output": raw}

        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    return {"title": "", "description": "Information about the query.",
            "narrative": "Relevant documents discuss the topic.",
            "raw_output": ""}

import json
print("generate_topic (OpenAI) definiert.")

generate_topic (OpenAI) definiert.


In [45]:
evidence_df_filtered = evidence_df[evidence_df["doc_variant"] == RUN_CONFIG["doc_variant"]].copy().reset_index(drop=True)

topic_rows = []

total_rows = len(evidence_df_filtered)

CHECKPOINT_EVERY = 200

for i, row in evidence_df_filtered.iterrows():
    print(
        f"{i+1}/{total_rows} | "
        f"{row['doc_variant']} | "
        f"{row['query_text'][:50]}"
    )

    topic = generate_topic(
        row["evidence"],
        model_provider=RUN_CONFIG["model_provider"],
        model=RUN_CONFIG["model"]
    )

    topic_rows.append({
        "session_id": row["session_id"],
        "doc_variant": row["doc_variant"],
        "query_text": row["query_text"],
        "tfidf_terms": row["tfidf_terms"],
        "evidence": row["evidence"],
        "title": topic["title"],
        "description": topic["description"],
        "narrative": topic["narrative"],
        "raw_output": topic["raw_output"]
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(topic_rows)
        checkpoint_path = f"data/topics_{RUN_CONFIG['run_name']}_checkpoint_{i+1}.csv"
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"Checkpoint gespeichert: {checkpoint_path}")

topics_df = pd.DataFrame(topic_rows)

import os
os.makedirs("data", exist_ok=True)
output_path = f"data/topics_{RUN_CONFIG['run_name']}.csv"
topics_df.to_csv(output_path, index=False)

print("Fertig:", len(topics_df))
print("Final gespeichert:", output_path)

1/376 | contrastive | peer to peer communication
2/376 | contrastive | wattpad reading comprehension
3/376 | contrastive | synthetic biology nasa
4/376 | contrastive | artificial intelligence in education
5/376 | contrastive | procrastination among students
6/376 | contrastive | cannabis
7/376 | contrastive | asphalt engine oil
8/376 | contrastive | Which feeding pattern most strongly predicts eligi
9/376 | contrastive | relationship between financial literacy, financial
10/376 | contrastive | theories about meco waste and coconut shell
11/376 | contrastive | balance in business profit and environmental susta
12/376 | contrastive | vitamin b12
13/376 | contrastive | virtual tour
14/376 | contrastive | the psychological factor affecting student entrepr
15/376 | contrastive | green roof
16/376 | contrastive | political science
17/376 | contrastive | mnemonic device agent based modeling
18/376 | contrastive | How can hands on projects and simulations be combi
19/376 | contrastive | commun

In [46]:
# Safety: topics_df aus CSV laden falls die Generation-Cell nicht frisch gelaufen ist
import pandas as pd
csv_path = f"data/topics_{RUN_CONFIG['run_name']}.csv"
if 'topics_df' not in dir() or len(topics_df) == 0:
    topics_df = pd.read_csv(csv_path)
    print(f"topics_df aus CSV geladen: {len(topics_df)} Zeilen")
else:
    print(f"topics_df aus Memory: {len(topics_df)} Zeilen")


topics_df aus Memory: 376 Zeilen


In [47]:
import json
import os

os.makedirs("runfiles", exist_ok=True)
output_path = f"runfiles/{RUN_CONFIG['run_name']}.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in topics_df.iterrows():
        entry = {
            "qid": str(row["session_id"]),
            "query": row["query_text"],
            "description": row["description"],
            "narrative": row["narrative"]
        }
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Gespeichert: {output_path}")
print(f"Anzahl Topics: {len(topics_df)}")

Gespeichert: runfiles/openai_first_contrastive_tfidf5.jsonl
Anzahl Topics: 376


In [48]:
import shutil
import os

# Alle 3 Runs sammeln und in runs/ Ordner kopieren
RUNS = [
    "openai_first_rel_tfidf5",
    "openai_first_non_rel_tfidf5",
    "openai_first_contrastive_tfidf5",
]

os.makedirs("runs", exist_ok=True)

missing = []
for run in RUNS:
    src = f"runfiles/{run}.jsonl"
    dst = f"runs/{run}.jsonl"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        with open(dst, encoding="utf-8") as f:
            count = sum(1 for _ in f)
        print(f"  {run}.jsonl  →  runs/  ({count} Topics)")
    else:
        missing.append(run)
        print(f"  FEHLT noch: {src}")

if missing:
    print(f"{len(missing)} Run(s) noch nicht fertig: {missing}")
else:
    print(f"Alle 3 Runs in runs/ gespeichert.")

  openai_first_rel_tfidf5.jsonl  →  runs/  (381 Topics)
  openai_first_non_rel_tfidf5.jsonl  →  runs/  (376 Topics)
  openai_first_contrastive_tfidf5.jsonl  →  runs/  (376 Topics)
Alle 3 Runs in runs/ gespeichert.
